# Прогон spellcheck (Qwen3.5-0.8B)

Инференс **Qwen3.5-0.8B** на тестовой выборке [`BW/RU_SPELLCHECK_DEVICE`](https://huggingface.co/datasets/BW/RU_SPELLCHECK_DEVICE) в двух режимах:
- **seed prompt** — короткий system-промпт из [`gepa_spellcheck_prompt.json`](../gepa_spellcheck_prompt.json);
- **с промптом** — system-промпт из [`prompt.txt`](../prompt.txt).

Результаты сохраняются в `seed_prompt_eval_predictions.csv` и `prompt_eval_predictions.csv`. Оценка **ERRANT** / **RUSpellEval** — скриптом [`evaluate_prompt_errant.py`](../evaluate_prompt_errant.py).

**Нужно:** GPU (Colab T4+).

**Runtime → Run all**

## 1. Окружение

In [1]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


## 2. Установка зависимостей

In [2]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec

import torch
torch._dynamo.config.recompile_limit = 64

## 3. Загрузка промпта

In [3]:
from pathlib import Path
import json

PROMPT_CANDIDATES = [
    Path("../prompt.txt"),
    Path("prompt.txt"),
    Path("/content/NLP_PROJECT/prompt.txt"),
    Path("/content/prompt.txt"),
]

PROMPT_PATH = next((p for p in PROMPT_CANDIDATES if p.is_file()), None)
if PROMPT_PATH is None:
    raise FileNotFoundError(
        "prompt.txt не найден. Загрузите файл или клонируйте репозиторий."
    )

SYSTEM_PROMPT = PROMPT_PATH.read_text(encoding="utf-8").strip()
print(f"Loaded prompt from: {PROMPT_PATH.resolve()}")
print(f"Prompt length: {len(SYSTEM_PROMPT)} chars")
print("\n--- Preview ---")
print(SYSTEM_PROMPT[:400] + ("..." if len(SYSTEM_PROMPT) > 400 else ""))

GEPA_PROMPT_CANDIDATES = [
    Path("../gepa_spellcheck_prompt.json"),
    Path("gepa_spellcheck_prompt.json"),
    Path("/content/NLP_PROJECT/gepa_spellcheck_prompt.json"),
    Path("/content/gepa_spellcheck_prompt.json"),
]

GEPA_PROMPT_PATH = next((p for p in GEPA_PROMPT_CANDIDATES if p.is_file()), None)
if GEPA_PROMPT_PATH is None:
    SEED_PROMPT = (
        "Ты корректор русского текста. Исправь орфографические, пунктуационные "
        "и регистровые ошибки в предложении пользователя. Верни только исправленный "
        "текст без пояснений и кавычек."
    )
else:
    SEED_PROMPT = json.loads(GEPA_PROMPT_PATH.read_text(encoding="utf-8"))["seed_prompt"]

print(f"\nSeed prompt source: {GEPA_PROMPT_PATH.resolve() if GEPA_PROMPT_PATH else 'built-in fallback'}")
print(f"Seed prompt length: {len(SEED_PROMPT)} chars")
print(SEED_PROMPT)

Loaded prompt from: /content/prompt.txt
Prompt length: 3169 chars

--- Preview ---
You are a Russian‑language proofreader.  
Your ONLY job is to correct **orthographic (spelling), punctuation, and capitalization** mistakes in the text supplied by the user.  
Everything else must stay exactly as it was: wording, word order, line breaks, spaces that are not part of an error, and any non‑text symbols (e.g., numbers, emojis) must be preserved.

### How to work

1. **Read the whole i...

Seed prompt source: built-in fallback
Seed prompt length: 175 chars
Ты корректор русского текста. Исправь орфографические, пунктуационные и регистровые ошибки в предложении пользователя. Верни только исправленный текст без пояснений и кавычек.


## 4. Тестовый датасет RU_SPELLCHECK_DEVICE

In [4]:
from datasets import load_dataset

DATASET_NAME = "BW/RU_SPELLCHECK_DEVICE"
test_df = load_dataset(DATASET_NAME, split="test").to_pandas().dropna()

sources = test_df["typed"].tolist()
corrections = test_df["original"].tolist()

print(f"Loaded {DATASET_NAME} (test): {len(test_df)} rows")
print(f"Columns: {list(test_df.columns)}")
print("\nExample:")
print(f"  typed:    {sources[0]}")
print(f"  original: {corrections[0]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/598 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/69.4k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/67.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/296 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/296 [00:00<?, ? examples/s]

Loaded BW/RU_SPELLCHECK_DEVICE (test): 296 rows
Columns: ['id', 'created_at', 'player_id', 'device_type', 'original', 'typed', '__index_level_0__']

Example:
  typed:    - Графини , - отвечал будочник.
  original: — Графини , — отвечал будочник.


## 5. Загрузка Qwen3.5-0.8B

In [6]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

QWEN_MODEL = "unsloth/Qwen3.5-0.8B"
MAX_SEQ_LENGTH = 1024

qwen_model, qwen_tokenizer = FastModel.from_pretrained(
    model_name=QWEN_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    full_finetuning=False,
)

qwen_tokenizer = get_chat_template(
    qwen_tokenizer,
    chat_template="qwen3-instruct",
)

Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

==((====))==  Unsloth 2026.7.2: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [7]:
import re


def strip_thinking_tags(text: str) -> str:
    text = re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    think_close = "</think>"
    if think_close in text:
        text = text.split(think_close, maxsplit=1)[-1]
    return text.strip()


def predict_qwen(
    sentences: list[str],
    system_prompt: str | None = SYSTEM_PROMPT,
) -> list[str]:
    predictions = []
    for sentence in sentences:
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": sentence})

        text = qwen_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = qwen_tokenizer(
            text=[text],
            images=None,
            videos=None,
            return_tensors="pt",
        ).to(qwen_model.device)

        input_len = inputs["input_ids"].shape[1]
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=max(int(len(sentence) * 1.5), 32),
            do_sample=False,
        )
        pred = strip_thinking_tags(
            qwen_tokenizer.decode(
                outputs[0][input_len:],
                skip_special_tokens=True,
            )
        )
        predictions.append(pred)
    return predictions


## 6. Инференс с seed prompt

In [22]:
demo_sentence = "И не чсно прохожим в этот день непогожйи почему я веселый такйо"
print("Demo (seed prompt):")
print(f"  input:  {demo_sentence}")
print(f"  output: {predict_qwen([demo_sentence], system_prompt=SEED_PROMPT)[0]}")

pred_qwen_seed = predict_qwen(sources, system_prompt=SEED_PROMPT)
print(f"Predictions (seed prompt): {len(pred_qwen_seed)}")


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


Demo (seed prompt):
  input:  И не чсно прохожим в этот день непогожйи почему я веселый такйо


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


  output: <think>
Thinking Process:

1.  **Analyze the Request:**
    *   **Role:** Text corrector (корректор русского текста).
    *   **Task:** Correct orthographic, punctuation, and typographical errors in the provided user sentence.
    *   **Output:** Only the corrected text (without explanations or quotes).
    *   **Input:** "И не чсно прохожим в этот день непогожйи


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

Predictions (seed prompt): 296


## 7. Сохранение предсказаний с seed prompt

In [23]:
from datetime import datetime, timezone

import pandas as pd

results_df_seed = pd.DataFrame({
    "typed": sources,
    "original": corrections,
    "prediction": pred_qwen_seed,
    "exact_match": [p == c for p, c in zip(pred_qwen_seed, corrections)],
    "model": QWEN_MODEL,
    "prompt_path": str(GEPA_PROMPT_PATH.resolve()) if GEPA_PROMPT_PATH else "seed_prompt",
    "dataset": DATASET_NAME,
    "split": "test",
    "created_at": datetime.now(timezone.utc).isoformat(),
})

print(
    f"Exact match (seed prompt): "
    f"{results_df_seed['exact_match'].mean():.2%} "
    f"({results_df_seed['exact_match'].sum()}/{len(results_df_seed)})"
)
print("\n--- Первые 5 примеров ---")
for i, row in results_df_seed.head(5).iterrows():
    print(f"\n[{i + 1}]")
    print(f"  typed:       {row['typed']}")
    print(f"  original:    {row['original']}")
    print(f"  prediction:  {row['prediction']}")
    print(f"  exact_match: {row['exact_match']}")


Exact match (seed prompt): 0.34% (1/296)

--- Первые 5 примеров ---

[1]
  typed:       - Графини , - отвечал будочник.
  original:    — Графини , — отвечал будочник.
  prediction:  <think>
Thinking Process:

1.  **Analyze the Request:**
    *   Role: Russian text editor (корректор русского текста).
    *   Task: Correct orthographic, punctuation, and typographical errors
  exact_match: False

[2]
  typed:       ялидавта Ивановна решилась отвкчать.
  original:    Лизавета Ивановна решилась отвечать.
  prediction:  <think>
Thinking Process:

1.  **Analyze the Request:**
    *   Role: Russian text editor (corregter).
    *   Task: Correct orthographic, punctuation, and typographical errors in the provided sentence.
    *
  exact_match: False

[3]
  typed:       Он шел;
  original:    Он шел;
  prediction:  <think>
Thinking Process:

1.  **Analyze the Request:**
    *   Role: Text corrector (корректор русского текста).
  exact_match: False

[4]
  typed:       вмдел в небе несчетные хвезды

In [24]:
SEED_OUTPUT_CANDIDATES = [
    Path("../seed_prompt_eval_predictions.csv"),
    Path("seed_prompt_eval_predictions.csv"),
    Path("/content/NLP_PROJECT/seed_prompt_eval_predictions.csv"),
    Path("/content/seed_prompt_eval_predictions.csv"),
]

SEED_OUTPUT_PATH = next(
    (p for p in SEED_OUTPUT_CANDIDATES if p.parent.exists()),
    Path("seed_prompt_eval_predictions.csv"),
)

results_df_seed.to_csv(SEED_OUTPUT_PATH, index=False)
print(f"Saved: {SEED_OUTPUT_PATH.resolve()}")
print("\nДля оценки ERRANT локально:")
print(f"  python evaluate_prompt_errant.py --input {SEED_OUTPUT_PATH.resolve()}")


Saved: /seed_prompt_eval_predictions.csv

Для оценки ERRANT локально:
  python evaluate_prompt_errant.py --input /seed_prompt_eval_predictions.csv


In [25]:
!cp /seed_prompt_eval_predictions.csv /content/seed_prompt_eval_predictions.csv

## 8. Инференс с промптом из prompt.txt

In [8]:
demo_sentence = "И не чсно прохожим в этот день непогожйи почему я веселый такйо"
print("Demo (с промптом из prompt.txt):")
print(f"  input:  {demo_sentence}")
print(f"  output: {predict_qwen([demo_sentence], system_prompt=SYSTEM_PROMPT)[0]}")

pred_qwen = predict_qwen(sources, system_prompt=SYSTEM_PROMPT)
print(f"Predictions (prompt.txt): {len(pred_qwen)}")

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


Demo (с промптом из prompt.txt):
  input:  И не чсно прохожим в этот день непогожйи почему я веселый такйо


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


  output: Не ччно прохожим в этот день непогожии, почему я веселый такто


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

Predictions (prompt.txt): 296


## 9. Сохранение предсказаний с промптом

In [10]:
from datetime import datetime, timezone

import pandas as pd

results_df = pd.DataFrame({
    "typed": sources,
    "original": corrections,
    "prediction": pred_qwen,
    "exact_match": [p == c for p, c in zip(pred_qwen, corrections)],
    "model": QWEN_MODEL,
    "prompt_path": str(PROMPT_PATH.resolve()),
    "dataset": DATASET_NAME,
    "split": "test",
    "created_at": datetime.now(timezone.utc).isoformat(),
})

print(f"Exact match: {results_df['exact_match'].mean():.2%} ({results_df['exact_match'].sum()}/{len(results_df)})")
print("\n--- Первые 5 примеров ---")
for i, row in results_df.head(5).iterrows():
    print(f"\n[{i + 1}]")
    print(f"  typed:       {row['typed']}")
    print(f"  original:    {row['original']}")
    print(f"  prediction:  {row['prediction']}")
    print(f"  exact_match: {row['exact_match']}")

Exact match: 14.19% (42/296)

--- Первые 5 примеров ---

[1]
  typed:       - Графини , - отвечал будочник.
  original:    — Графини , — отвечал будочник.
  prediction:  - Графини, отвечал будочник.
  exact_match: False

[2]
  typed:       ялидавта Ивановна решилась отвкчать.
  original:    Лизавета Ивановна решилась отвечать.
  prediction:  Ялидвата Ивановна решилась отвкчать.
  exact_match: False

[3]
  typed:       Он шел;
  original:    Он шел;
  prediction:  Он шел;
  exact_match: True

[4]
  typed:       вмдел в небе несчетные хвезды, светившие его путь и какк лев выступал сильно и бодро, так что когда восходящее солнце озарило своими влажнозкрасными лучами только что расходившегося молодца, мужду Москвой и им легко уже тридцать пять верст. Через два дня он уже был дома, в своей избенке, к велимому изумлению солдатки, которую туда поселили.
  original:    видел в небе несчетные звезды, светившие его путь, и как лев выступал сильно и бодро, так что когда восходящее солнце озарило 

In [11]:
OUTPUT_CANDIDATES = [
    Path("../prompt_eval_predictions.csv"),
    Path("prompt_eval_predictions.csv"),
    Path("/content/NLP_PROJECT/prompt_eval_predictions.csv"),
    Path("/content/prompt_eval_predictions.csv"),
]

OUTPUT_PATH = next(
    (p for p in OUTPUT_CANDIDATES if p.parent.exists()),
    Path("prompt_eval_predictions.csv"),
)

results_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH.resolve()}")
print("\nДля оценки ERRANT локально:")
print(f"  python evaluate_prompt_errant.py --input {OUTPUT_PATH.resolve()}")

Saved: /prompt_eval_predictions.csv

Для оценки ERRANT локально:
  python evaluate_prompt_errant.py --input /prompt_eval_predictions.csv


In [12]:
!cp /prompt_eval_predictions.csv /content/prompt_eval_predictions.csv
!cp /seed_prompt_eval_predictions.csv /content/seed_prompt_eval_predictions.csv

cp: cannot stat '/seed_prompt_eval_predictions.csv': No such file or directory
